# 🎨 Animagine XL 3.1 - Free Unlimited Google Colab GPU Backend

This notebook turns a **free Google Colab T4 GPU (16GB VRAM)** into your own private image generation server.

### Why use this?
- 🚀 **Zero Quota Limits**: No 90-second Hugging Face ZeroGPU limit.
- 🆓 **100% Free**: Uses Google's free T4 GPU tier.
- ⚡ **Fast & Full Quality**: Generates authentic anime 16:9 images in ~3-5 seconds.

### Instructions:
1. In the top menu, go to **Runtime** > **Change runtime type** > Select **T4 GPU**.
2. Run the cells below.
3. Copy the public URL generated at the end: `Running on public URL: https://xxxx.gradio.live`
4. Paste it into your local `.env` file:
   ```env
   ANIMAGINE_URL="https://xxxx.gradio.live"
   ```
5. Now run your local scripts (`python images/main.py` or `python pipeline/main.py`) — it will automatically route through your Colab GPU for unlimited generations!

### Step 1: Install Dependencies

In [ ]:
!pip install -q diffusers transformers accelerate invisible-watermark gradio peft safetensors torchvision

### Step 2: Load Model & Launch Private Gradio API Backend

In [ ]:
import os
import torch
import traceback
import gradio as gr
from diffusers import StableDiffusionXLPipeline, EulerAncestralDiscreteScheduler

print("⏳ Loading Animagine XL 3.1 model into Colab GPU VRAM...")
model_id = "cagliostrolab/animagine-xl-3.1"

pipe = StableDiffusionXLPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    use_safetensors=True,
)
pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)
pipe.to("cuda")

# T4 VRAM optimizations (prevents OOM on high resolutions)
pipe.enable_attention_slicing()
try:
    pipe.enable_vae_tiling()
except Exception:
    pass

print("✔ Animagine XL 3.1 successfully loaded on GPU!")

# Negative prompt preset
DEFAULT_NEGATIVE = "low quality, worst quality, deformed, bad anatomy, bad hands, missing fingers, extra digits, blurry, mutated, text, watermark"

def generate_api(
    prompt: str,
    negative_prompt: str = DEFAULT_NEGATIVE,
    seed: int = 42,
    width: int = 1344,
    height: int = 768,
    guidance_scale: float = 7.0,
    steps: int = 20,
    sampler: str = "Euler a",
    aspect_ratio: str = "1344 x 768",
    style_preset: str = "Anime",
    quality_preset: str = "Standard v3.1",
    use_upscaler: bool = False,
    strength: float = 0.5,
    upscale_by: float = 1.2,
    add_quality_tags: bool = True
):
    try:
        cur_seed = int(seed) if seed is not None else 42
        generator = torch.Generator("cuda").manual_seed(cur_seed)
        
        # Append anime quality tags
        final_prompt = prompt or ""
        if add_quality_tags:
            final_prompt = f"masterpiece, best quality, {final_prompt}"

        image = pipe(
            prompt=final_prompt,
            negative_prompt=negative_prompt,
            width=int(width),
            height=int(height),
            guidance_scale=float(guidance_scale),
            num_inference_steps=int(steps),
            generator=generator
        ).images[0]

        out_path = f"/content/output_{cur_seed}.png"
        image.save(out_path, format="PNG")
        return [out_path], {"seed": cur_seed}
    except Exception as e:
        traceback.print_exc()
        raise e

# Expose exact same Gradio schema matching the HF space
with gr.Blocks(title="Animagine XL 3.1 Private Colab Backend") as demo:
    gr.Markdown("# 🎨 Animagine XL 3.1 Private Backend (Colab T4 GPU)")
    prompt_in = gr.Textbox(label="Prompt", value="1girl, hatsune miku, masterpiece, 16:9")
    neg_in = gr.Textbox(label="Negative Prompt", value=DEFAULT_NEGATIVE)
    seed_in = gr.Number(label="Seed", value=42)
    width_in = gr.Number(label="Width", value=1344)
    height_in = gr.Number(label="Height", value=768)
    scale_in = gr.Number(label="Guidance Scale", value=7.0)
    steps_in = gr.Number(label="Steps", value=20)
    sampler_in = gr.Textbox(label="Sampler", value="Euler a")
    ar_in = gr.Textbox(label="Aspect Ratio", value="1344 x 768")
    style_in = gr.Textbox(label="Style", value="Anime")
    quality_in = gr.Textbox(label="Quality", value="Standard v3.1")
    upscaler_in = gr.Checkbox(label="Use Upscaler", value=False)
    strength_in = gr.Number(label="Strength", value=0.5)
    upscale_by_in = gr.Number(label="Upscale By", value=1.2)
    add_tags_in = gr.Checkbox(label="Add Quality Tags", value=True)

    gallery_out = gr.Gallery(label="Output")
    meta_out = gr.JSON(label="Metadata")

    btn = gr.Button("Generate")
    btn.click(
        fn=generate_api,
        inputs=[prompt_in, neg_in, seed_in, width_in, height_in, scale_in, steps_in, sampler_in, ar_in, style_in, quality_in, upscaler_in, strength_in, upscale_by_in, add_tags_in],
        outputs=[gallery_out, meta_out],
        api_name="run"
    )

print("\n🚀 Launching Public API Tunnel (Share URL)...\n")
demo.queue().launch(share=True, show_error=True)
